In [14]:
import re
import pandas as pd
import dotenv
import os
from neo4j import GraphDatabase

In [4]:
triplets_df_path = "../data/TEST_TRIPLETS_gfmag_sustainable.csv"
env_file = "../Neo4j_private.txt"

# clear neo4j 

In [16]:
dotenv.load_dotenv(NEO4J_ENV)

True

In [17]:
from neo4j import GraphDatabase

dotenv.load_dotenv(env_file)
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USER, PWD))

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

driver.close()

# ------

In [5]:
df = pd.read_csv(triplets_df_path)

In [13]:
df.loc[
    (df['entity1_type'] == 'country') |
    (df['entity2_type'] == 'country')
]

,entity1,entity1_type,rel_type,entity2,entity2_type,sector,url,date,original_rel,original_sector,original_e1,original_e2
4,China,country,is_member_of,developing countries,other,public_sector,https://gfmag.com/sustainable-finance/coal-tak...,2025-11-09T09:30:00+00:00,is_member_of,public_sector,China:country,developing countries:sector
5,India,country,is_member_of,developing countries,other,public_sector,https://gfmag.com/sustainable-finance/coal-tak...,2025-11-09T09:30:00+00:00,is_member_of,public_sector,India:country,developing countries:sector
6,China,country,is_competitor_of,India,country,public_sector,https://gfmag.com/sustainable-finance/coal-tak...,2025-11-09T09:30:00+00:00,is_competitor_of,public_sector,China:country,India:country
7,China,country,has_positive_impact,renewables,financial_instrument,energy,https://gfmag.com/sustainable-finance/coal-tak...,2025-11-09T09:30:00+00:00,has_positive_impact,energy,China:country,renewables:financial_instrument
10,Tunisia,country,relates_to,new legislation,product_service,public_sector,https://gfmag.com/executive-interviews/biat-ce...,2025-11-07T10:16:56+00:00,enacts,public_sector,Tunisia:country,new legislation:product_service
...,...,...,...,...,...,...,...,...,...,...,...,...
1261,Brazil,country,has_negative_impact,globalization,other,financials,https://gfmag.com/economics-policy-regulation/...,2013-12-02T00:00:00+00:00,has_negative_impact,financials,Brazil:country,globalization:concept
1263,China,country,has_negative_impact,Great Recession of 2008–2009,disaster_event,financials,https://gfmag.com/economics-policy-regulation/...,2013-12-02T00:00:00+00:00,has_negative_impact,financials,China:country,Great Recession of 2008–2009:disaster_event
1264,India,country,has_negative_impact,Great Recession of 2008–2009,disaster_event,financials,https://gfmag.com/economics-policy-regulation/...,2013-12-02T00:00:00+00:00,has_negative_impact,financials,India:country,Great Recession of 2008–2009:disaster_event
1265,UK,country,relates_to,sovereign sukuk,financial_instrument,financials,https://gfmag.com/banking/uk-a-capital-of-isla...,2013-12-02T00:00:00+00:00,issues,financials,UK:country,sovereign sukuk:financial_instrument


In [18]:
import pycountry

countries = list(pycountry.countries)

In [35]:
pycountry.countries.lookup("US")

Country(alpha_2='US', alpha_3='USA', flag='🇺🇸', name='United States', numeric='840', official_name='United States of America')

In [44]:
import re
import pycountry

def normalize_entity(name: str, entity_type: str):
    # clean name (punctuation + whitespace)
    if isinstance(name, str):
        cleaned = re.sub(r"[^\w\s]", "", name)
        cleaned = re.sub(r"\s+", " ", cleaned).strip()
    else:
        cleaned = name

    # try country lookup
    try:
        pycountry.countries.lookup(cleaned)
        is_country = True
    except Exception:
        is_country = False

    # case handling
    if is_country:
        return cleaned, "country"
    if entity_type == "country":
        return cleaned, "other"
    return cleaned, entity_type


In [45]:
df[["entity1", "entity1_type"]] = df.apply(
    lambda r: normalize_entity(r.entity1, r.entity1_type),
    axis=1,
    result_type="expand"
)

df[["entity2", "entity2_type"]] = df.apply(
    lambda r: normalize_entity(r.entity2, r.entity2_type),
    axis=1,
    result_type="expand"
)